In [1]:
# ==========================================================
# ACTIVIDAD 15. Cálculo de las 5 variables de riesgo
# Ventanas móviles hacia atrás (solo información hasta cada
# día), dentro de cada acción. Marca las filas utilizables
# tras la separación de 63 sesiones entre bloques.
# Todavía NO recorta extremos ni escala (Actividad 16).
# ==========================================================
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

VENTANA_CORTA = 21      # ~ 1 mes
VENTANA_LARGA = 63      # ~ 1 trimestre (también es la separación entre bloques)
VENTANA_VOLUMEN = 5     # ~ 1 semana
ANUALIZAR = np.sqrt(252)

drive.mount("/content/drive")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Precios de la muestra (Actividad 14)
# ----------------------------------------------------------
df = pd.read_parquet(CARPETA_DATOS / "precios_muestra.parquet",
                     columns=["ticker", "date", "close", "adj_close", "volume", "grupo", "bloque"])
df["ticker"] = df["ticker"].astype(str)
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
print("Filas:", len(df), "| empresas:", df["ticker"].nunique())

def movil(serie, ventana, funcion):
    """Aplica una ventana móvil dentro de cada acción (nunca mezcla empresas)."""
    rolling = serie.groupby(df["ticker"]).rolling(ventana, min_periods=ventana)
    return getattr(rolling, funcion)().reset_index(level=0, drop=True)

# ----------------------------------------------------------
# 2. Ingredientes: rendimiento diario y volumen en dólares
# ----------------------------------------------------------
df["r"] = np.log(df["adj_close"]).groupby(df["ticker"]).diff()
df["dolares"] = df["close"] * df["volume"]          # invariante a splits y contrasplits

# ----------------------------------------------------------
# 3. Las 5 variables
# ----------------------------------------------------------
vol_corta = movil(df["r"], VENTANA_CORTA, "std") * ANUALIZAR
vol_larga = movil(df["r"], VENTANA_LARGA, "std") * ANUALIZAR

df["volatilidad_21"] = vol_corta
df["rendimiento_21"] = movil(df["r"], VENTANA_CORTA, "sum")
df["caida_maximo_63"] = df["adj_close"] / movil(df["adj_close"], VENTANA_LARGA, "max") - 1
with np.errstate(divide="ignore", invalid="ignore"):   # evita avisos por divisiones entre 0
    df["cambio_volatilidad"] = np.log(vol_corta / vol_larga)
    df["volumen_anormal"] = np.log(movil(df["dolares"], VENTANA_VOLUMEN, "mean")
                                   / movil(df["dolares"], VENTANA_LARGA, "median"))
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]

# Un logaritmo de 0 da -infinito (por ejemplo, una semana entera sin volumen): se marca como nulo
infinitos = np.isinf(df[VARIABLES]).sum()
df[VARIABLES] = df[VARIABLES].replace([np.inf, -np.inf], np.nan)

# ----------------------------------------------------------
# 4. Filas utilizables: separación de 63 sesiones en cada bloque
# ----------------------------------------------------------
# Posición de cada sesión dentro de su bloque (0, 1, 2, ...). Las primeras 63
# de cada bloque usarían precios del bloque anterior (o no tienen historia).
df["posicion_bloque"] = df.groupby(["ticker", "bloque"], observed=True).cumcount()
df["utilizable"] = df["posicion_bloque"] >= VENTANA_LARGA

print("\n=== PRIMERA FECHA UTILIZABLE Y SESIONES POR BLOQUE ===")
print(df[df["utilizable"]].groupby("bloque", observed=True)["date"]
      .agg(primera="min", ultima="max", sesiones="nunique"))

print("\n=== FILAS UTILIZABLES: grupo x bloque ===")
print(pd.crosstab(df.loc[df["utilizable"], "grupo"], df.loc[df["utilizable"], "bloque"],
                  margins=True, margins_name="Total"))

util = df[df["utilizable"]]
print("\nInfinitos convertidos en nulos (todas las filas):")
print(infinitos)
print("\nNulos en las filas utilizables:")
print(util[VARIABLES].isna().sum())

# ----------------------------------------------------------
# 5. Primera mirada: empresas y periodo de entrenamiento
# ----------------------------------------------------------
ent = util[(util["grupo"] == "entrenamiento") & (util["bloque"] == "entrenamiento")]
print("\n=== DISTRIBUCIÓN (empresas de entrenamiento, bloque de entrenamiento) ===")
print(ent[VARIABLES].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

print("\n=== CORRELACIONES ENTRE VARIABLES (mismo subconjunto) ===")
print(ent[VARIABLES].corr().round(2))

print("\n=== COMPROBACIÓN: los 5 días con la volatilidad mediana más alta ===")
mediana_diaria = ent.groupby("date")[["volatilidad_21", "caida_maximo_63"]].median()
print(mediana_diaria.sort_values("volatilidad_21", ascending=False).head(5))

# ----------------------------------------------------------
# 6. Guardar solo las filas utilizables
# ----------------------------------------------------------
salida = util[["ticker", "date", "grupo", "bloque"] + VARIABLES].reset_index(drop=True)
for col in ["ticker", "grupo", "bloque"]:
    salida[col] = salida[col].astype("category")
salida.to_parquet(CARPETA_DATOS / "variables_muestra.parquet", index=False)
print("\nGuardado: variables_muestra.parquet -", len(salida), "filas")

Mounted at /content/drive
Filas: 2726974 | empresas: 1591

=== PRIMERA FECHA UTILIZABLE Y SESIONES POR BLOQUE ===
                 primera     ultima  sesiones
bloque                                       
entrenamiento 2020-02-05 2022-12-30       733
prueba        2025-04-04 2026-08-31       353
validacion    2023-04-04 2024-12-31       439

=== FILAS UTILIZABLES: grupo x bloque ===
bloque         entrenamiento  prueba  validacion    Total
grupo                                                    
entrenamiento         815829  392889      488607  1697325
reservada             350374  168734      209842   728950
Total                1166203  561623      698449  2426275

Infinitos convertidos en nulos (todas las filas):
volatilidad_21        0
cambio_volatilidad    0
rendimiento_21        0
caida_maximo_63       0
volumen_anormal       0
dtype: int64

Nulos en las filas utilizables:
volatilidad_21        0
cambio_volatilidad    0
rendimiento_21        0
caida_maximo_63       0
volumen_an